AAI Construction
----------------
The AAI captures public health administrative exposure as the z-score of
log-transformed FQHC service delivery sites per 100,000 residents:

    AAI = z-score( log(1 + FQHC_sites_per_100k) )

Design decisions:
  - Log transformation: addresses right skew (CA alone has 3,000+ sites)
  - Z-score: enables interpretation in standard deviation units
  - Exclusion of demographic variables from index: foreign-born share and
    uninsured rate are used ONLY as regression controls, not as index
    components. Including them in the index would create a construct
    validity problem (same variable on both sides of the regression).

Interaction term:
    AAI_x_foreign_born = AAI × foreign_born_share
Tests whether the AAI effect is stronger in high-immigration states.

Policy controls (binary, state-level):
  - border_state: TX, CA, AZ, NM
  - state_287g: states with ICE 287(g) enforcement agreements
  - sanctuary_policy: states with formal sanctuary designations
  
Source: ILRC and ICE public records

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "shared"))

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from config import (
    DATA_INTERIM, DATA_PROCESSED,
    BORDER_STATES, STATE_287G, SANCTUARY,
)


def zscore(series: pd.Series) -> pd.Series:
    clean = series.replace([np.inf, -np.inf], np.nan).fillna(0).to_frame()
    return pd.Series(
        StandardScaler().fit_transform(clean).ravel(),
        index=series.index
    )


def main():
    acs  = pd.read_csv(DATA_INTERIM / "acs_state_controls.csv",    dtype={"state_fips": str})
    ice  = pd.read_csv(DATA_INTERIM / "ice_state_outcomes.csv",    dtype={"state_fips": str})
    hrsa = pd.read_csv(DATA_INTERIM / "hrsa_state_visibility.csv", dtype={"state_fips": str})

    df = (
        acs
        .merge(ice,  on="state_fips", how="left")
        .merge(hrsa, on="state_fips", how="left")
    )

    # Coerce numerics
    for col in ["arrests", "detainers", "fqhc_site_count"]:
        df[col] = pd.to_numeric(df.get(col, 0), errors="coerce").fillna(0)

    # Rate variables
    pop = df["population"].replace(0, np.nan)
    df["fqhc_sites_per_100k"]  = 100_000 * df["fqhc_site_count"] / pop
    df["arrests_per_100k"]     = 100_000 * df["arrests"]          / pop
    df["detainers_per_100k"]   = 100_000 * df["detainers"]        / pop

    # AAI construction
    df["log_aai"] = np.log1p(df["fqhc_sites_per_100k"].fillna(0))
    df["aai"]     = zscore(df["log_aai"])
    df["aai_x_foreign_born"] = df["aai"] * df["foreign_born_share"].fillna(0)

    # Policy controls
    df["border_state"]     = df["state_abbr"].isin(BORDER_STATES).astype(int)
    df["state_287g"]       = df["state_abbr"].isin(STATE_287G).astype(int)
    df["sanctuary_policy"] = df["state_abbr"].isin(SANCTUARY).astype(int)
    df["has_287g"]         = df["state_287g"]

    out = DATA_PROCESSED / "state_analysis_table.csv"
    df.to_csv(out, index=False)
    print(f"Saved → {out}  ({len(df)} states, {len(df.columns)} columns)")

    cols = ["NAME","aai","arrests_per_100k","detainers_per_100k",
            "fqhc_sites_per_100k","foreign_born_share","state_287g"]
    print("\nTop 10 by arrest rate:")
    print(df[cols].sort_values("arrests_per_100k", ascending=False)
          .head(10).to_string(index=False))


if __name__ == "__main__":
    main()